# Defeating Context Rot: Recursive Retrieval for Long-Document QA

This notebook presents the results of our experiments comparing Recursive Language Model (RLM) querying against baselines for long-document question answering.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.analyze import full_analysis, print_summary
from analysis.plots import (
    plot_needle_heatmap,
    plot_needle_by_length,
    plot_needle_by_position,
    plot_multihop_comparison,
    plot_longbench_comparison,
    plot_overall_summary,
)
from src.cost_tracker import tracker

sns.set_theme(style='whitegrid', font_scale=1.2)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## 1. Problem Statement

**Context rot** refers to the accuracy degradation when relevant facts are buried deep within long input contexts. As LLM context windows grow (100K–1M+ tokens), models increasingly struggle to attend to information in the middle of long documents.

We investigate whether **Recursive Language Model (RLM) querying** — a controller-based approach that recursively decomposes questions and retrieves targeted evidence — can mitigate context rot compared to:
- **Full Context**: Sending the entire document to the LLM
- **RAG**: Single-pass retrieve-and-generate
- **Map-Reduce**: Map over chunks, reduce answers

## 2. Method: RLM Controller

The RLM algorithm:
1. **SEARCH** the document for relevant chunks
2. **READ** the top chunks
3. **REASON** about the question with the evidence, producing an answer + confidence score
4. If confidence < threshold: **DECOMPOSE** into sub-questions, **RECURSE**, and **MERGE**

Key design choices:
- Hybrid retrieval (BM25 + vector search via Reciprocal Rank Fusion)
- Controller-based architecture (LLM as oracle, not autonomous agent)
- Structured JSON output for confidence extraction

## 3. Results

In [ ]:
results = full_analysis()
print_summary(results)

### 3.1 Needle-in-Haystack

This benchmark measures how well each method retrieves a single fact ("needle") embedded at various positions within documents of varying lengths.

In [ ]:
if 'needle' in results:
    display(results['needle']['overall'])

In [ ]:
plot_needle_heatmap()
from IPython.display import Image
Image(filename='../results/plots/needle_heatmap.png')

In [ ]:
plot_needle_by_length()
Image(filename='../results/plots/needle_by_length.png')

In [ ]:
plot_needle_by_position()
Image(filename='../results/plots/needle_by_position.png')

### 3.2 Multi-Hop QA

Multi-hop questions require chaining 2-3 facts scattered across the document.

In [ ]:
if 'multihop' in results:
    display(results['multihop']['overall'])

In [ ]:
plot_multihop_comparison()
Image(filename='../results/plots/multihop_comparison.png')

### 3.3 LongBench (QASPER + NarrativeQA)

Real-world long-document QA from academic papers (QASPER) and narrative texts (NarrativeQA).

In [ ]:
if 'longbench' in results:
    display(results['longbench']['overall'])

In [ ]:
plot_longbench_comparison()
Image(filename='../results/plots/longbench_comparison.png')

### 3.4 Overall Summary

In [ ]:
plot_overall_summary()
Image(filename='../results/plots/overall_summary.png')

## 4. Cost Analysis

In [ ]:
try:
    with open('../results/cost_summary.txt') as f:
        print(f.read())
except FileNotFoundError:
    print(tracker.summary())

## 5. Conclusions

Key findings:

1. **Context rot is real**: Full-context performance degrades as document length increases and when facts are positioned in the middle of documents.

2. **RLM mitigates context rot**: By recursively decomposing questions and retrieving targeted evidence, RLM maintains more consistent performance across document lengths and needle positions.

3. **Multi-hop advantage**: RLM shows the largest gains on multi-hop questions where single-pass methods struggle to chain multiple facts.

4. **Cost-accuracy tradeoff**: RLM uses more API calls (and thus more budget) per question, but the accuracy gains justify the cost for complex queries.

5. **Hybrid retrieval matters**: The combination of BM25 and vector search via RRF provides robust retrieval across different question types.